In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Dataset Loading

In [2]:
products = pd.read_csv('pharmacy_products.csv') 
batches = pd.read_csv('pharmacy_inventory_batches.csv') 
sales = pd.read_csv('pharmacy_sales_transactions.csv')

In [3]:
products.head()

,product_id,product_name,category,unit_cost_usd,selling_price_usd,supplier_lead_time_days,reorder_level_units
0,PRD001,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50
1,PRD002,Azithromycin 250mg,Antibiotics,11.21,21.18,21,300
2,PRD003,Ciprofloxacin 500mg,Antibiotics,40.31,55.21,10,30
3,PRD004,Cephalexin 500mg,Antibiotics,2.80,4.13,14,200
4,PRD005,Doxycycline 100mg,Antibiotics,2.65,3.87,21,300


In [4]:
batches.head()

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp


In [5]:
sales.head()

,transaction_id,transaction_date,product_id,batch_id,units_sold,unit_price_usd,total_revenue_usd,clearance_discount_applied
0,TXN-10001,2025-10-01,PRD020,BAT-2025-176,7,46.40,324.80,0
1,TXN-10002,2025-10-01,PRD042,BAT-2025-284,7,11.91,83.37,0
2,TXN-10003,2025-10-01,PRD023,BAT-2025-198,3,27.71,83.13,0
3,TXN-10004,2025-10-01,PRD038,BAT-2025-267,8,46.58,372.64,0
4,TXN-10005,2025-10-01,PRD034,BAT-2025-245,1,22.04,22.04,0


In [6]:
sales["batch_id"].nunique()

193

## Checking for missing values 

In [7]:
products.isnull().sum()

product_id                 0
product_name               0
category                   0
unit_cost_usd              0
selling_price_usd          0
supplier_lead_time_days    0
reorder_level_units        0
dtype: int64

In [8]:
batches.isnull().sum()

batch_id                  0
product_id                0
received_date             0
expiry_date               0
quantity_received         0
current_stock_quantity    0
storage_type              0
dtype: int64

In [9]:
sales.isnull().sum()

transaction_id                0
transaction_date              0
product_id                    0
batch_id                      0
units_sold                    0
unit_price_usd                0
total_revenue_usd             0
clearance_discount_applied    0
dtype: int64

##  Converting Date Fields to Datetime

In [10]:
batches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   batch_id                193 non-null    object
 1   product_id              193 non-null    object
 2   received_date           193 non-null    object
 3   expiry_date             193 non-null    object
 4   quantity_received       193 non-null    int64 
 5   current_stock_quantity  193 non-null    int64 
 6   storage_type            193 non-null    object
dtypes: int64(2), object(5)
memory usage: 10.7+ KB


In [11]:
batches['received_date'] = pd.to_datetime(batches['received_date'])
batches['expiry_date'] = pd.to_datetime(batches['expiry_date']) 
sales['transaction_date'] = pd.to_datetime(sales['transaction_date'])

In [12]:
batches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   batch_id                193 non-null    object        
 1   product_id              193 non-null    object        
 2   received_date           193 non-null    datetime64[ns]
 3   expiry_date             193 non-null    datetime64[ns]
 4   quantity_received       193 non-null    int64         
 5   current_stock_quantity  193 non-null    int64         
 6   storage_type            193 non-null    object        
dtypes: datetime64[ns](2), int64(2), object(3)
memory usage: 10.7+ KB


In [13]:
batches.head()

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp


In [14]:
# Set Current Analysis Date 
analysis_date = pd.to_datetime('2026-10-10')

### Feature engineering

In [15]:
## adding new column to the batches dataset
batches['days_to_expiry'] = (batches['expiry_date'] - analysis_date).dt.days

In [16]:
batches.head()

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type,days_to_expiry
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp,207
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage,-3
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage,-180
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp,-59
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp,97


In [17]:
conditions = [
    batches['days_to_expiry'] < 0,
    (batches['days_to_expiry'] >=0) &
(batches['days_to_expiry'] <=30),
    (batches['days_to_expiry'] > 30) &
(batches['days_to_expiry'] <= 60),
    (batches['days_to_expiry'] > 60) &
(batches['days_to_expiry'] <= 90),
    batches['days_to_expiry'] > 90
]

choices = [
    'Expired',
    'Critical (0-30 Days)',
    'High Risk (31-60 Days)',
    'Moderate Risk (61-90 Days)',
    'Fresh Stock (90 Days)'
]

batches['expiry_status'] = np.select(conditions,choices,default='Unknown')

In [18]:
batches.head(15)

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type,days_to_expiry,expiry_status
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp,207,Fresh Stock (90 Days)
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage,-3,Expired
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage,-180,Expired
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp,-59,Expired
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp,97,Fresh Stock (90 Days)
5,BAT-2025-106,PRD002,2025-12-15,2027-06-08,200,188,Cold Storage,241,Fresh Stock (90 Days)
6,BAT-2025-107,PRD002,2025-05-30,2026-02-24,500,15,Cold Storage,-228,Expired
7,BAT-2025-108,PRD002,2026-06-19,2027-12-11,300,158,Room Temp,427,Fresh Stock (90 Days)
8,BAT-2025-109,PRD003,2026-06-18,2026-12-15,500,206,Room Temp,66,Moderate Risk (61-90 Days)
9,BAT-2025-110,PRD003,2026-06-07,2027-03-04,200,98,Room Temp,145,Fresh Stock (90 Days)


## merging dataset (batches and product)

In [19]:
master_inventory = batches.merge(products, on='product_id', how='left')

In [20]:
master_inventory.head()

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type,days_to_expiry,expiry_status,product_name,category,unit_cost_usd,selling_price_usd,supplier_lead_time_days,reorder_level_units
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp,207,Fresh Stock (90 Days),Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage,-3,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage,-180,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp,-59,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp,97,Fresh Stock (90 Days),Azithromycin 250mg,Antibiotics,11.21,21.18,21,300


In [21]:
master_inventory['stock_value_cost_usd'] = master_inventory['current_stock_quantity'] * master_inventory['unit_cost_usd']

master_inventory['stock_value_retail_usd'] = master_inventory['current_stock_quantity'] * master_inventory['selling_price_usd']

In [22]:
master_inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   batch_id                 193 non-null    object        
 1   product_id               193 non-null    object        
 2   received_date            193 non-null    datetime64[ns]
 3   expiry_date              193 non-null    datetime64[ns]
 4   quantity_received        193 non-null    int64         
 5   current_stock_quantity   193 non-null    int64         
 6   storage_type             193 non-null    object        
 7   days_to_expiry           193 non-null    int64         
 8   expiry_status            193 non-null    object        
 9   product_name             193 non-null    object        
 10  category                 193 non-null    object        
 11  unit_cost_usd            193 non-null    float64       
 12  selling_price_usd        193 non-nul

In [23]:
master_inventory.head()

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type,days_to_expiry,expiry_status,product_name,category,unit_cost_usd,selling_price_usd,supplier_lead_time_days,reorder_level_units,stock_value_cost_usd,stock_value_retail_usd
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp,207,Fresh Stock (90 Days),Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,20963.80,27670.50
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage,-3,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,2433.56,3212.10
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage,-180,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,879.60,1161.00
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp,-59,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,1319.40,1741.50
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp,97,Fresh Stock (90 Days),Azithromycin 250mg,Antibiotics,11.21,21.18,21,300,425.98,804.84


### Sales velocity

In [24]:
last_90_days = analysis_date - pd.Timedelta(days=90)

recent_sales = sales[sales['transaction_date'] >= last_90_days]

In [25]:
sales_velocity = recent_sales.groupby('product_id').agg(
    total_units_sold_90d=('units_sold', 'sum'),
    avg_daily_sales=('units_sold', lambda x: x.sum() / 90.0)
).reset_index()

In [26]:
# Merge Sales Velocity into Master Inventory
master_inventory = master_inventory.merge(sales_velocity, on='product_id', how='left')

master_inventory['avg_daily_sales'] = master_inventory['avg_daily_sales'].fillna(0)

In [27]:
master_inventory.head()

,batch_id,product_id,received_date,expiry_date,quantity_received,current_stock_quantity,storage_type,days_to_expiry,expiry_status,product_name,category,unit_cost_usd,selling_price_usd,supplier_lead_time_days,reorder_level_units,stock_value_cost_usd,stock_value_retail_usd,total_units_sold_90d,avg_daily_sales
0,BAT-2025-101,PRD001,2025-11-11,2027-05-05,1000,715,Room Temp,207,Fresh Stock (90 Days),Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,20963.80,27670.50,223.0,2.477778
1,BAT-2025-102,PRD001,2025-04-15,2026-10-07,100,83,Cold Storage,-3,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,2433.56,3212.10,223.0,2.477778
2,BAT-2025-103,PRD001,2025-10-15,2026-04-13,750,30,Cold Storage,-180,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,879.60,1161.00,223.0,2.477778
3,BAT-2025-104,PRD001,2026-02-13,2026-08-12,100,45,Room Temp,-59,Expired,Amoxicillin 500mg,Antibiotics,29.32,38.70,7,50,1319.40,1741.50,223.0,2.477778
4,BAT-2025-105,PRD002,2026-07-19,2027-01-15,300,38,Room Temp,97,Fresh Stock (90 Days),Azithromycin 250mg,Antibiotics,11.21,21.18,21,300,425.98,804.84,313.0,3.477778


In [28]:
### Calculate Estimated Days of Stock Remaining
master_inventory['days_of_stock_remaining'] = np.where(
    master_inventory['avg_daily_sales'] > 0,
    master_inventory['current_stock_quantity'] / master_inventory['avg_daily_sales'],
    9999
)

In [29]:
### Flag Batches Guaranteed to Expire Before Selling Out
master_inventory['waste_risk_flag'] = (
    (master_inventory['days_to_expiry'] > 0) & 
    (master_inventory['days_to_expiry'] <= 90) & 
    (master_inventory['days_of_stock_remaining'] > master_inventory['days_to_expiry'])
)

In [30]:
### Export Cleaned Master Dataset for Excel & Power BI 
master_inventory.to_csv('cleaned_master_inventory.csv', index=False)

# EDA(EXPLORATORY DATA ANALYSIS)....